# 课程笔记

## 1. 神经网络优化与驻点

训练可能在损失仍然较大时停止改进：

$$
L(\theta_t) \text{ 较大}, \qquad \|\nabla_\theta L(\theta_t)\| \approx 0.
$$

此时梯度下降几乎不更新：

$$
\theta_{t+1}=\theta_t-\eta\nabla_\theta L(\theta_t) \approx \theta_t.
$$

这通常是优化瓶颈，不一定是过拟合。

驻点满足

$$
\nabla_\theta L(\theta')=0.
$$

驻点包括局部最小值、局部最大值和鞍点。鞍点附近同时存在上升方向和下降方向。

## 2. Hessian 矩阵

对二阶可微标量函数 $f: \mathbb{R}^n \to \mathbb{R}$，Hessian 定义为

$$
H_f(\mathbf{x}) = \nabla^2 f(\mathbf{x}), \qquad (H_f)_{ij} = \frac{\partial^2 f}{\partial x_i \partial x_j}.
$$

对损失函数 $L$，在 $\theta'$ 附近的二阶 Taylor 近似为

$$
L(\theta) \approx L(\theta') + g^T(\theta-\theta') + \frac{1}{2}(\theta-\theta')^T H(\theta-\theta'),
$$

其中

$$
g=\nabla_\theta L(\theta'), \qquad H=\nabla_\theta^2 L(\theta').
$$

在驻点处 $g=0$。令 $v=\theta-\theta'$，则

$$
L(\theta)-L(\theta') \approx \frac{1}{2}v^T H v.
$$

设 $\lambda_i$ 为 $H$ 的特征值：

- 对所有 $i$，$\lambda_i>0$：局部最小值。
- 对所有 $i$，$\lambda_i<0$：局部最大值。
- 正、负特征值同时存在：鞍点。

若 $Hu=\lambda u$ 且 $\lambda<0$，则

$$
u^T H u=\lambda\|u\|^2<0.
$$

对很小的 $\epsilon>0$，

$$
L(\theta'+\epsilon u)-L(\theta') \approx \frac{1}{2}\epsilon^2\lambda\|u\|^2<0.
$$

因此即使梯度为零，负特征值方向仍可降低损失。

## 3. Hessian 例子

若

$$
f(x,y)=3x^2+xy+y^2,
$$

则

$$
H_f=
\begin{bmatrix}
6 & 1 \\
1 & 2
\end{bmatrix}.
$$

由于 $H_f\succ0$，$(0,0)$ 为局部最小值。

对一个极简网络

$$
\hat y=w_1w_2x, \qquad x=1, \qquad y=1,
$$

取

$$
L(w_1,w_2)=(w_1w_2-1)^2.
$$

在 $(w_1,w_2)=(0,0)$，

$$
\nabla L(0,0)=0, \qquad
H(0,0)=
\begin{bmatrix}
0 & -2 \\
-2 & 0
\end{bmatrix}.
$$

特征值为 $2$ 和 $-2$，故原点是鞍点。

## 4. 高维损失曲面

现代深度网络中，完整 Hessian 特征分解通常代价过高。但 Hessian 视角说明：高维空间中严格的坏局部最小值较少见。

定义 minimum ratio：

$$
\text{minimum ratio}=\frac{\#\{\lambda_i>0\}}{\#\{\lambda_i\}}.
$$

若比例为 $1$，所有局部曲率为正。若比例小于 $1$，仍存在负曲率方向。高维网络中，高损失停滞点更常是鞍点或平坦区域。

## 5. Batch 与 Momentum

设数据量为 $N$，batch size 为 $B$。Full-batch 使用

$$
L(\theta)=\frac{1}{N}\sum_{n=1}^N \ell_n(\theta), \qquad B=N.
$$

Mini-batch 使用随机近似

$$
L_{\mathcal{B}}(\theta)=\frac{1}{B}\sum_{i\in\mathcal{B}}\ell_i(\theta), \qquad B<N.
$$

不同 batch 对应不同局部目标。即使

$$
\nabla L_{\mathcal{B}_1}(\theta)\approx0,
$$

通常仍有

$$
\nabla L_{\mathcal{B}_2}(\theta)\ne0.
$$

因此 mini-batch 噪声可帮助参数离开驻点或尖锐区域。小 batch 通常偏向平坦极小值；大 batch 更接近确定性下降，可能进入尖锐极小值。

Momentum 累积历史梯度：

$$
m_{t+1}=\lambda m_t-\eta g_t, \qquad \theta_{t+1}=\theta_t+m_{t+1}, \qquad g_t=\nabla L(\theta_t).
$$

若 $m_0=0$，

$$
m_{t+1}=-\eta\sum_{\tau=0}^t\lambda^{t-\tau}g_\tau.
$$

更新方向是历史梯度的指数加权和，可平滑噪声并保持稳定运动。

## 6. 自适应学习率

大梯度不一定带来有效下降。在狭长谷中，更新可能在陡峭方向震荡。可使用参数级缩放：

$$
\theta_{t+1,i}=\theta_{t,i}-\frac{\eta_t}{\sigma_{t,i}}g_{t,i}.
$$

Adagrad 使用所有历史梯度平方：

$$
\sigma_{t,i}=\sqrt{\frac{1}{t+1}\sum_{\tau=0}^t g_{\tau,i}^2}.
$$

大梯度坐标步长变小，小梯度坐标步长变大。

RMSProp 使用指数移动平均：

$$
\sigma_{t,i}^2=\alpha\sigma_{t-1,i}^2+(1-\alpha)g_{t,i}^2, \qquad 0\le\alpha<1.
$$

分母更依赖近期曲率。

Adam 结合 Momentum 与 RMSProp：

$$
m_t\approx \text{EMA}(g_t), \qquad \sigma_t^2\approx \text{EMA}(g_t^2), \qquad \theta_{t+1}=\theta_t-\eta_t\frac{m_t}{\sigma_t}.
$$

学习率调度控制全局尺度 $\eta_t$。常见衰减为

$$
\eta_t=\frac{\eta_0}{\sqrt{t+1}}.
$$

Warmup 先用较小 $\eta_t$，再增大并衰减，可降低早期统计不稳定的影响。

## 7. 分类损失

对 $K$ 分类，标签用 one-hot 向量表示：

$$
\hat y\in\{e_1,\dots,e_K\}.
$$

给定 logits $z\in\mathbb{R}^K$，softmax 为

$$
p_i=\frac{e^{z_i}}{\sum_{j=1}^K e^{z_j}}, \qquad p_i\in(0,1), \qquad \sum_i p_i=1.
$$

常见损失：

$$
L_{\mathrm{MSE}}=\sum_{i=1}^K(p_i-\hat y_i)^2,
$$

$$
L_{\mathrm{CE}}=-\sum_{i=1}^K\hat y_i\log p_i.
$$

若真实类别为 $c$，则

$$
L_{\mathrm{CE}}=-\log p_c.
$$

softmax + cross entropy 有

$$
\frac{\partial L_{\mathrm{CE}}}{\partial z_i}=p_i-\hat y_i.
$$

模型很自信但错误时，交叉熵仍保留较大有效梯度。softmax 后接 MSE 容易产生平坦区域，使优化更困难。

## 8. Batch Normalization

若特征尺度差异很大，线性模型

$$
y=w_1x_1+w_2x_2+b
$$

具有不均匀敏感度：

$$
\Delta y=\Delta w_i x_i.
$$

大尺度输入产生陡峭方向，小尺度输入产生平坦方向，损失等高线被拉长，优化不稳定。

对一个 batch 中的激活 $z^1,\dots,z^B$，

$$
\mu_{\mathcal{B}}=\frac{1}{B}\sum_{b=1}^B z^b, \qquad
\sigma_{\mathcal{B}}^2=\frac{1}{B}\sum_{b=1}^B(z^b-\mu_{\mathcal{B}})^2.
$$

标准化并恢复可学习尺度：

$$
\tilde z^b=\frac{z^b-\mu_{\mathcal{B}}}{\sqrt{\sigma_{\mathcal{B}}^2+\epsilon}}, \qquad
\hat z^b=\gamma\tilde z^b+\beta.
$$

$\gamma$ 与 $\beta$ 为可学习参数。

推理时使用训练中的移动统计量：

$$
\bar\mu\leftarrow(1-p)\bar\mu+p\mu_{\mathcal{B}}, \qquad
\bar\sigma^2\leftarrow(1-p)\bar\sigma^2+p\sigma_{\mathcal{B}}^2.
$$

Batch normalization 的主要作用是平滑优化曲面并允许更大的学习率，而不只是稳定隐藏层分布。

## 9. 卷积神经网络

图像是张量

$$
X\in\mathbb{R}^{H\times W\times C}.
$$

展平后 $x\in\mathbb{R}^D$，其中 $D=HWC$。若全连接层有 $M$ 个隐藏单元，参数量为

$$
MD+M.
$$

高分辨率图像会带来巨大模型。

CNN 加入图像结构偏置：

- 局部感受野：每个神经元只看 $k_h\times k_w\times C_{\mathrm{in}}$ 的局部 patch；
- 权重共享：同一 filter 在不同空间位置复用。

对输入 $X\in\mathbb{R}^{H\times W\times C_{\mathrm{in}}}$ 和 filters $W\in\mathbb{R}^{k\times k\times C_{\mathrm{in}}\times C_{\mathrm{out}}}$，

$$
Y_{i,j,c}=\sigma\left(\sum_{m=1}^{k}\sum_{n=1}^{k}\sum_{d=1}^{C_{\mathrm{in}}}X_{is+m,js+n,d}W_{m,n,d,c}+b_c\right).
$$

输出 $Y$ 是 feature map。堆叠小卷积核可以扩大有效感受野，同时保持较少参数。

Pooling 是固定下采样算子。Max pooling 为

$$
Y_{i,j,c}=\max_{(m,n)\in\Omega_{i,j}}X_{m,n,c}.
$$

它降低空间分辨率与计算量，但可能丢失局部细节。

## 10. 邻域嵌入

流形学习假设高维样本 $x_i\in\mathbb{R}^D$ 接近某个低维流形。目标是嵌入

$$
x_i\mapsto z_i\in\mathbb{R}^d, \qquad d\ll D,
$$

并保留局部邻域几何。

LLE 先用邻居 $N(i)$ 重构每个点：

$$
\min_W\sum_i\left\|x_i-\sum_{j\in N(i)}w_{ij}x_j\right\|^2,
\qquad \sum_{j\in N(i)}w_{ij}=1.
$$

再寻找保持相同局部权重的低维点：

$$
\min_Z\sum_i\left\|z_i-\sum_{j\in N(i)}w_{ij}z_j\right\|^2.
$$

LLE 保留局部重构，但没有显式拉开非邻居。

Laplacian Eigenmaps 构建加权图 $W=(w_{ij})$，并最小化

$$
\frac{1}{2}\sum_{i,j}w_{ij}\|z_i-z_j\|^2=\operatorname{tr}(Z^TLZ),
\qquad L=D-W,
$$

约束为

$$
Z^TDZ=I, \qquad D_{ii}=\sum_jw_{ij}.
$$

解来自最小的非平凡广义特征向量：

$$
Lv=\lambda Dv.
$$

t-SNE 将点对相似度转为概率分布。原空间中，

$$
p_{j|i}=\frac{\exp(-\|x_i-x_j\|^2/2\sigma_i^2)}{\sum_{k\ne i}\exp(-\|x_i-x_k\|^2/2\sigma_i^2)},
\qquad p_{ij}=\frac{p_{j|i}+p_{i|j}}{2N}.
$$

嵌入空间中使用长尾 Student 分布：

$$
q_{ij}=\frac{(1+\|z_i-z_j\|^2)^{-1}}{\sum_k\sum_{m\ne k}(1+\|z_k-z_m\|^2)^{-1}}.
$$

优化目标为

$$
\mathrm{KL}(P\|Q)=\sum_i\sum_jp_{ij}\log\frac{p_{ij}}{q_{ij}}.
$$

长尾分布使中等距离仍有概率质量，有助于分离簇并缓解拥挤。

## 11. 自注意力机制

对可变长序列

$$
X=[x_1;\dots;x_L]\in\mathbb{R}^{L\times d},
$$

self-attention 映射为

$$
X\mapsto O=[o_1;\dots;o_L]\in\mathbb{R}^{L\times d_v},
$$

其中每个 $o_i$ 可使用所有 $x_j$ 的信息。

投影：

$$
Q=XW^Q,\qquad K=XW^K,\qquad V=XW^V,
$$

$$
W^Q,W^K\in\mathbb{R}^{d\times d_k},\qquad W^V\in\mathbb{R}^{d\times d_v}.
$$

点积相关度：

$$
S=\frac{QK^T}{\sqrt{d_k}},\qquad S_{ij}=\frac{q_i^Tk_j}{\sqrt{d_k}}.
$$

按行归一化：

$$
A=\operatorname{softmax}(S),\qquad
\alpha_{ij}=\frac{\exp(S_{ij})}{\sum_{m=1}^{L}\exp(S_{im})},\qquad
\sum_{j=1}^{L}\alpha_{ij}=1.
$$

输出：

$$
O=AV,\qquad o_i=\sum_{j=1}^{L}\alpha_{ij}v_j.
$$

Softmax 可替换为其他非线性映射，但 softmax 给出归一化注意力权重。完整 attention 的复杂度为

$$
O(L^2d_k),
$$

因为需要比较所有 token pair。

Multi-head attention 使用多个子空间：

$$
\operatorname{head}_h=\operatorname{softmax}\left(\frac{Q_hK_h^T}{\sqrt{d_h}}\right)V_h,
$$

$$
\operatorname{MHA}(X)=\operatorname{Concat}(\operatorname{head}_1,\dots,\operatorname{head}_H)W^O.
$$

不同 head 可刻画不同关系，例如语法依赖或语义依赖。

Self-attention 具有置换等变性。对置换矩阵 $P$，

$$
\operatorname{SA}(PX)=P\operatorname{SA}(X),
$$

因此需要注入位置信息：

$$
\tilde{x}_i=x_i+p_i.
$$

$p_i$ 可固定或学习，可表示绝对位置或相对位置。

模型对比：

- CNN：局部、固定感受野；局域性归纳偏置强；小数据更稳定。
- RNN：顺序状态转移；难以并行；长距离信息可能衰减。
- Self-attention：全局、自适应感受野；位置维度可并行；归纳偏置弱，通常更依赖数据量。

常见变体：

$$
	ext{local attention:}\quad A_{ij}=0\quad 	ext{if } |i-j|>r,
$$

用于长语音序列时，可将复杂度近似降为 $O(Lr)$。

对图输入 $G=(V,E)$，mask 非邻接节点：

$$
A=\operatorname{softmax}(S+M),\qquad
M_{ij}=\begin{cases}
0,&(i,j)\in E,\\
-\infty,&(i,j)\notin E,
\end{cases}
$$

即 graph attention。高效 Transformer 变体通过近似 attention 将 $O(L^2)$ 降向 $O(L)$ 或 $O(L\log L)$，通常需要在精度与速度间折中。


## 12. Transformer Seq2seq

Seq2seq 将输入序列映射为长度未知的输出序列：

$$
X=(x_1,\dots,x_N),\qquad Y=(y_1,\dots,y_M),
$$

其中 $N$ 已知，$M$ 由模型生成到结束符为止。典型任务包括语音识别、机器翻译、摘要与句法分析。

Transformer encoder 映射为

$$
X\mapsto H=(h_1,\dots,h_N),\qquad H\in\mathbb{R}^{N\times d}.
$$

对 encoder block 输入 $A\in\mathbb{R}^{N\times d}$，

$$
Q=AW^Q,\qquad K=AW^K,\qquad V=AW^V.
$$

Scaled dot-product attention 为

$$
\operatorname{Attn}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V.
$$

$\sqrt{d_k}$ 用于避免点积过大导致 softmax 饱和。Multi-head attention 使用多个独立子空间：

$$
\operatorname{head}_i=\operatorname{Attn}(QW_i^Q,KW_i^K,VW_i^V),
$$

$$
\operatorname{MHA}(Q,K,V)=\operatorname{Concat}(\operatorname{head}_1,\dots,\operatorname{head}_h)W^O.
$$

每个 sublayer 使用残差连接与 layer normalization：

$$
\operatorname{Output}=\operatorname{LayerNorm}(x+\operatorname{Sublayer}(x)).
$$

Batch normalization 在 mini-batch 的单个特征维度上计算统计量：

$$
\mu_j=\frac{1}{B}\sum_{b=1}^{B}z_{b,j},
$$

Layer normalization 在单个样本的隐藏维度上计算统计量：

$$
\mu_b=\frac{1}{D}\sum_{d=1}^{D}z_{b,d}.
$$

LN 与 batch size 无关，更适合变长序列。Position-wise feed-forward network 在所有位置共享：

$$
\operatorname{FFN}(x)=\max(0,xW_1+b_1)W_2+b_2.
$$

Autoregressive decoder 将条件分布分解为

$$
P(Y\mid X)=\prod_{t=1}^{M}P(y_t\mid y_{<t},X).
$$

从 $y_0=\langle\mathrm{BOS}\rangle$ 开始，

$$
P(y_t\mid y_{0:t-1},X)=\operatorname{softmax}(\operatorname{logits}_t),
$$

当 $y_t=\langle\mathrm{EOS}\rangle$ 时停止。

Masked self-attention 保证因果性。令 $S=QK^T/\sqrt{d_k}$，

$$
M_{ij}=\begin{cases}
0,& i\ge j,\\
-\infty,& i<j,
\end{cases}
$$

$$
\operatorname{MaskedAttn}(Q,K,V)=\operatorname{softmax}(S+M)V.
$$

未来位置的注意力权重因此为零。Cross attention 将 encoder 信息注入 decoder：

$$
Q=Z_{\mathrm{dec}}W^Q,\qquad K=HW^K,\qquad V=HW^V,
$$

$$
\operatorname{CrossAttn}=\operatorname{softmax}\left(\frac{Q_{\mathrm{dec}}K_{\mathrm{enc}}^T}{\sqrt{d_k}}\right)V_{\mathrm{enc}}.
$$

给定目标序列 $\hat{Y}=(\hat{y}_1,\dots,\hat{y}_M)$，训练最小化平均交叉熵：

$$
\mathcal{L}=-\frac{1}{M}\sum_{t=1}^{M}\sum_{i=1}^{|\mathcal{V}|}\hat{y}_{t,i}\log y'_{t,i}.
$$

Teacher forcing 在训练时输入真实前缀：

$$
\text{decoder input at }t: \quad (\langle\mathrm{BOS}\rangle,\hat{y}_1,\dots,\hat{y}_{t-1}),
$$

而不是模型自己生成的前缀。它使优化更稳定，但带来 exposure bias：

$$
\text{train input}\sim p_{\mathrm{data}},\qquad \text{test input}\sim p_\theta.
$$

Scheduled sampling 混合两种输入来源：

$$
z_{t-1}=\begin{cases}
\hat{y}_{t-1},&\text{with probability }p,\\
y_{t-1}^{\mathrm{model}},&\text{with probability }1-p,
\end{cases}
\qquad p\downarrow 0.
$$

Greedy decoding 使用

$$
y_t=\arg\max_y P(y\mid y_{<t},X),
$$

只保证局部最优。Beam search 保留 log probability 最高的 $B$ 条部分序列：

$$
\operatorname{Score}(y_{1:t})=\sum_{\tau=1}^{t}\log P(y_\tau\mid y_{<\tau},X).
$$

Beam search 适合机器翻译等确定性任务；开放式生成常用 temperature sampling：

$$
P_T(y_t=i)=\frac{\exp(l_i/T)}{\sum_j\exp(l_j/T)}.
$$
